In [3]:
# %%
# =============================================================================
# Paper C · 03_gnn_reproduce  (10-seed version)
# Experiment condition ② : heterogeneous GNN on the enterprise knowledge graph
#
# A modern-PyG reimplementation of the ComRisk-style hetero GNN. Not a bit-exact
# reproduction of Wei et al. 2024 — an EQUIVALENT baseline measuring the uplift
# of graph structure over the 02 basic-info floor. Now run over 10 seeds
# (mean ± std) to match 05/06.
#
# Metrics: AUC / F1 / KS on test, mask A (strict) primary, mask B reference.
# Inputs : data/processed/smesd_hetero.pt   (person-fixed)
# Outputs: artifacts/tables/03_gnn_reproduce.csv
# =============================================================================
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "artifacts" / "tables"
MODELS = ROOT / "artifacts" / "models"
TABLES.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")   # no GPU needed; graph is tiny (~4k nodes)
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]   # 10 seeds (match 05/06)


# %%
# --- 1. Load graph (person-fixed version) ------------------------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
assert data["person"].num_nodes == 2405, (
    "person nodes != 2405 — re-run the fixed 01 cell 8 + save (cell 9) first")
data = data.to(DEVICE)
print(data)

# Standardize company features on TRAIN rows only (no leakage), same as 02.
from sklearn.preprocessing import StandardScaler
Xc = data["company"].x.numpy().astype(np.float64)
Xc[:, :2] = np.log1p(np.clip(Xc[:, :2], 0, None))     # capitals: same log as 02
tr_mask = data["company"].train_mask.numpy()
scaler = StandardScaler().fit(Xc[tr_mask])
data["company"].x = torch.tensor(scaler.transform(Xc), dtype=torch.float32)


# %%
# --- 2. Model: HeteroConv (per-relation SAGEConv) x2 + classifier ------------
from torch_geometric.nn import HeteroConv, SAGEConv

EDGE_TYPES = list(data.edge_types)
IN_DIMS = {"company": data["company"].x.size(1),   # 23
           "person": data["person"].x.size(1)}     # 32


class HeteroGNN(nn.Module):
    def __init__(self, hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.drop = dropout
        self.lin_in = nn.ModuleDict(
            {nt: nn.Linear(d, hidden) for nt, d in IN_DIMS.items()})
        self.convs = nn.ModuleList()
        for _ in range(layers):
            conv = HeteroConv(
                {et: SAGEConv((hidden, hidden), hidden) for et in EDGE_TYPES},
                aggr="sum")
            self.convs.append(conv)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, x_dict, edge_index_dict):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in x_dict.items()}
        for conv in self.convs:
            h = conv(h, edge_index_dict)
            h = {nt: F.relu(v) for nt, v in h.items()}
            h = {nt: F.dropout(v, p=self.drop, training=self.training)
                 for nt, v in h.items()}
        return self.head(h["company"])


# %%
# --- 3. Metric helpers (same as 02) ------------------------------------------
from sklearn.metrics import roc_auc_score, f1_score, roc_curve

def ks_stat(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.max(tpr - fpr))

def metrics(y, s, thr=0.5):
    return {"AUC": roc_auc_score(y, s),
            "F1": f1_score(y, (s >= thr).astype(int), zero_division=0),
            "KS": ks_stat(y, s)}


# %%
# --- 4. Train / evaluate one mask set at one seed ----------------------------
def run(mask_prefix, seed, epochs=200, lr=5e-3, wd=5e-4, patience=30):
    torch.manual_seed(seed)          # per-seed seeding inside run()
    np.random.seed(seed)
    comp = data["company"]
    if mask_prefix == "A":
        tr = comp.train_mask; va = comp.val_mask; te = comp.test_mask
    else:
        tr = comp.train_mask_repro; va = comp.val_mask_repro; te = comp.test_mask_repro
    y = comp.y

    cls = torch.bincount(y[tr], minlength=2).float()
    w = (cls.sum() / (2 * cls.clamp(min=1))).to(DEVICE)

    model = HeteroGNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    x_dict = {nt: data[nt].x for nt in data.node_types}
    ei_dict = {et: data[et].edge_index for et in EDGE_TYPES}

    best_val, best_state, wait = -1, None, 0
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        logits = model(x_dict, ei_dict)
        loss = F.cross_entropy(logits[tr], y[tr], weight=w)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            prob = F.softmax(model(x_dict, ei_dict), dim=1)[:, 1].cpu().numpy()
        val_auc = roc_auc_score(y[va].cpu().numpy(), prob[va.cpu().numpy()])
        if val_auc > best_val:
            best_val = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        prob = F.softmax(model(x_dict, ei_dict), dim=1)[:, 1].cpu().numpy()

    out = {}
    for split_name, m in [("val", va), ("test", te)]:
        mi = m.cpu().numpy()
        out[split_name] = metrics(y.cpu().numpy()[mi], prob[mi])
    return out, model


# %%
# --- 5. Run both mask sets × 10 seeds → mean ± std ---------------------------
rows = []
last_model_A = None
for mask_prefix in ["A", "B"]:
    for seed in SEEDS:
        res, model = run(mask_prefix, seed)
        if mask_prefix == "A":
            last_model_A = model
        for split_name, met in res.items():
            rows.append({"mask": mask_prefix, "model": "HeteroGNN",
                         "split": split_name, "seed": seed,
                         **{k: round(v, 4) for k, v in met.items()}})
    print(f"mask {mask_prefix}: done ({len(SEEDS)} seeds)")

raw = pd.DataFrame(rows)
raw.to_csv(TABLES / "03_gnn_reproduce_raw.csv", index=False)

# aggregate: mean ± std across seeds
results = (raw.groupby(["mask", "model", "split"])
           .agg(AUC_mean=("AUC", "mean"), AUC_std=("AUC", "std"),
                F1_mean=("F1", "mean"), F1_std=("F1", "std"),
                KS_mean=("KS", "mean"), KS_std=("KS", "std"))
           .round(4).reset_index())
print(results.to_string(index=False))
torch.save(last_model_A.state_dict(), MODELS / "03_gnn_maskA.pt")


# %%
# --- 6. Compare against the 02 floor -----------------------------------------
floor = pd.read_csv(TABLES / "02_baseline_basic.csv")
floor_A_test = floor[(floor["mask"] == "A") & (floor["split"] == "test")]
gnn_A_test = results[(results["mask"] == "A") & (results["split"] == "test")]

print("\n=== UPLIFT: graph (03) vs basic-info floor (02), mask A, test ===")
print("floor (02):")
print(floor_A_test[["model", "AUC"]].to_string(index=False))
print("graph (03), 10-seed mean ± std:")
print(gnn_A_test[["AUC_mean", "AUC_std"]].to_string(index=False))
best_floor_auc = floor_A_test["AUC"].max()
gnn_auc = gnn_A_test["AUC_mean"].iloc[0]
print(f"\nΔAUC (GNN - best floor) = {gnn_auc - best_floor_auc:+.4f}")


# %%
# --- 7. Persist + notes ------------------------------------------------------
results.to_csv(TABLES / "03_gnn_reproduce.csv", index=False)
print(f"saved: 03_gnn_reproduce.csv (10-seed mean ± std)")
print("""
Condition ② established (equivalent hetero-GNN, modern PyG, 10 seeds)
====================================================================
  NOT a bit-exact Wei et al. 2024 reproduction — an equivalent multi-relation
  hetero-GNN baseline. Report ΔAUC over the 02 floor as evidence that graph
  structure (lawsuit relations + contagion) helps (RQ1).
""")
print("03_gnn_reproduce complete.")

C:\Users\miy\miniconda3\envs\credit_override_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HeteroData(
  company={
    x=[3976, 23],
    y=[3976],
    train_mask=[3976],
    val_mask=[3976],
    test_mask=[3976],
    train_mask_repro=[3976],
    val_mask_repro=[3976],
    test_mask_repro=[3976],
    num_nodes=3976,
  },
  person={
    x=[2405, 32],
    num_nodes=2405,
  },
  (company, rel_0, person)={
    edge_index=[2, 698],
    edge_weight=[698],
  },
  (person, rel_1, company)={
    edge_index=[2, 698],
    edge_weight=[698],
  },
  (company, rel_2, person)={
    edge_index=[2, 883],
    edge_weight=[883],
  },
  (person, rel_3, company)={
    edge_index=[2, 883],
    edge_weight=[883],
  },
  (company, rel_6, company)={
    edge_index=[2, 938],
    edge_weight=[938],
  },
  (company, rel_7, company)={
    edge_index=[2, 942],
    edge_weight=[942],
  },
  (company, rel_9, person)={
    edge_index=[2, 3657],
    edge_weight=[3657],
  },
  (person, rel_8, company)={
    edge_index=[2, 3657],
    edge_weight=[3657],
  },
  (company, rel_10, company)={
    edge_index=[2, 376

In [6]:
# %%
# --- [FIG] RQ1: basic-info floor -> hetero-GNN uplift (10-seed) --------------
# 02 floor: LogReg is deterministic (single value). 03 GNN: 10-seed mean ± std.
# %%
# --- Figure export setup: create paper_C folder + save helper (run once) -----
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PAPER_DIR = ROOT / "paper_C"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

# Embed fonts as text (avoids font substitution at typesetting). EPS has no alpha.
matplotlib.rcParams.update({
    "ps.fonttype": 42, "pdf.fonttype": 42, "axes.unicode_minus": False,
})

def save_fig(fig, name, dpi=600):
    """Save as PDF + EPS + PNG (caption-free). Does not print the path."""
    for ext in ("pdf", "eps", "png"):
        fig.savefig(PAPER_DIR / f"{name}.{ext}", format=ext, dpi=dpi,
                    bbox_inches="tight")
    print(f"saved: {name}.pdf / .eps / .png")

print("figure export ready")

import numpy as np, pandas as pd, matplotlib.pyplot as plt

TABLES = ROOT / "artifacts" / "tables"
floor = pd.read_csv(TABLES / "02_baseline_basic.csv")
gnn = pd.read_csv(TABLES / "03_gnn_reproduce.csv")   # now has AUC_mean / AUC_std

f_at = floor[(floor["mask"] == "A") & (floor["split"] == "test")]
floor_auc = f_at["AUC"].max()
floor_model = f_at.loc[f_at["AUC"].idxmax(), "model"]

g_at = gnn[(gnn["mask"] == "A") & (gnn["split"] == "test")]
gnn_auc = g_at["AUC_mean"].iloc[0]
gnn_std = g_at["AUC_std"].iloc[0]

fig, ax = plt.subplots(figsize=(4.6, 4.2))
x = np.arange(2)
ax.bar(x, [floor_auc, gnn_auc], width=0.55,
       yerr=[0, gnn_std], capsize=5,
       color=["#9aa0a6", "#c0392b"], edgecolor="black", linewidth=0.8)
for xi, v in zip(x, [floor_auc, gnn_auc]):
    ax.text(xi, v + 0.012, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
ax.annotate("", xy=(1, gnn_auc), xytext=(1, floor_auc),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.2))
ax.text(1.28, (floor_auc + gnn_auc) / 2, f"+{gnn_auc - floor_auc:.3f}",
        ha="left", va="center", fontsize=10, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([f"Basic-info floor\n({floor_model})",
                    "Hetero-GNN\n(lawsuit + relations)"], fontsize=9)
ax.set_ylabel("Test AUC (mask A)", fontsize=10)
ax.set_ylim(0.50, 0.85)
ax.grid(axis="y", ls=":", alpha=0.5)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
plt.tight_layout()
save_fig(fig, "fig_rq1_uplift")
plt.close(fig)

figure export ready


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


saved: fig_rq1_uplift.pdf / .eps / .png


In [2]:
# %%
# --- Check whether SMEsD.md ships a codebook for cause/court/category --------
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
REPO = ROOT / "data" / "external" / "ComRisk"

# list repo docs and print SMEsD.md if present
for p in sorted(REPO.glob("*")):
    print(p.name)

md = REPO / "SMEsD.md"
if md.exists():
    print("\n===== SMEsD.md =====\n")
    print(md.read_text(encoding="utf-8", errors="replace"))
else:
    print("\nSMEsD.md not found at repo root; searching recursively...")
    for m in REPO.rglob("*.md"):
        print(f"  found: {m.relative_to(REPO)}")

.git
data
gnn.py
metapath2vec
model_save
README.md
sample.py
SMEsD.md
train.py
utils.py

===== SMEsD.md =====

# SMEsD (V1.0) (update on 2024-02-05)
## Introduction
The dataset is used in paper "Combining intra-risk and contagion risk for enterprise bankruptcy prediction using graph neural networks" [Paper](https://www.sciencedirect.com/science/article/abs/pii/S0020025523016675?dgcid=author)

The SMEsD consists of 3,976 SMEs and related persons in China from 2014 to 2021, which constitutes a multiplex enterprise knowledge graph. All enterprises are associated with their basic business information and lawsuit events spanning from 2000 to 2021. Specifically, the enterprise business information includes registered capital, paid-in capital and established time. Each lawsuit consists of the associated plaintiff, defendant, subjects, court level, result and timestamp.

We split the SMEsD into train, valid and test dataset according to the enterprise bankruptcy time, i.e., 2014-2018 for train